# Add Trips to Gold

## Import Packages

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    BooleanType,
    DateType,
    TimestampType,
    DecimalType
)

In [0]:
from pyspark import pipelines as dp

## Variables

## Schema Definition

In [0]:
schema = StructType([
    StructField(
        name="vendor_id",
        dataType=IntegerType(),
        nullable=True,
        metadata={
            "comment": "Identifier of the technology provider that processed and submitted the trip record"
        }
    ),
    StructField(
        name="pickup_datetime",
        dataType=TimestampType(),
        nullable=True,
        metadata={
            "comment": "Date and time when the taxi meter was engaged and the trip officially started"
        }
    ),
    StructField(
        name="dropoff_datetime",
        dataType=TimestampType(),
        nullable=True,
        metadata={
            "comment": "Date and time when the taxi meter was disengaged and the trip ended"
        }
    ),
    StructField(
        name="day_of_trip",
        dataType=DateType(),
        nullable=True,
        metadata={
            "comment": "Date and time when the taxi meter was disengaged and the trip ended"
        }
    ),
    StructField(
        name="passengers",
        dataType=IntegerType(),
        nullable=True,
        metadata={
            "comment": "Number of passengers in the vehicle during the trip"
        }
    ),
    StructField(
        name="distance",
        dataType=DecimalType(10,2),
        nullable=True,
        metadata={
            "comment": "Distance traveled during the trip in miles as reported by the taximeter"
        }
    ),
    StructField(
        name="duration",
        dataType=DecimalType(10,5),
        nullable=True,
        metadata={
            "comment": "Duration traveled during the trip in minutes calculated by pickup and dropoff timestamps"
        }
    ),
    StructField(
        name="rate_code_id",
        dataType=IntegerType(),
        nullable=True,
        metadata={
            "comment": "Rate code applied to the trip, including standard, JFK, Newark, negotiated fare, or group ride"
        }
    ),
    StructField(
        name="rate_code_description",
        dataType=StringType(),
        nullable=True,
        metadata={"comment":"Shows the Rate Code in Plain English"}
    ),
    StructField(
        name="store_and_fwd_flag",
        dataType=StringType(),
        nullable=True,
        metadata={
            "comment": "Indicates whether the trip record was stored locally before transmission due to communication interruptions"
        }
    ),
    StructField(
        name="pickup_location_id",
        dataType=IntegerType(),
        nullable=True,
        metadata={
            "comment": "TLC Taxi Zone identifier where the passenger was picked up"
        }
    ),
    StructField(
        name="dropoff_location_id",
        dataType=IntegerType(),
        nullable=True,
        metadata={
            "comment": "TLC Taxi Zone identifier where the passenger was dropped off"
        }
    ),
    StructField(
        name="payment_type",
        dataType=IntegerType(),
        nullable=True,
        metadata={
            "comment": "Payment method used by the passenger, such as credit card, cash, no charge, dispute, or unknown"
        }
    ),
    StructField(
        name="payment_description",
        dataType=StringType(),
        nullable=True,
        metadata={"comment":"Shows the Payment Code in Plain English"}
    ),
    StructField(
        name="fare_amount",
        dataType=DecimalType(10,2),
        nullable=True,
        metadata={
            "comment": "Base fare charged for the trip before taxes, tolls, surcharges, and tips"
        }
    ),
    StructField(
        name="extra",
        dataType=DecimalType(10,2),
        nullable=True,
        metadata={
            "comment": "Additional surcharges applied to the fare, including peak-hour and overnight charges"
        }
    ),
    StructField(
        name="mta_tax",
        dataType=DecimalType(10,2),
        nullable=True,
        metadata={
            "comment": "Mandatory tax collected on behalf of the Metropolitan Transportation Authority"
        }
    ),
    StructField(
        name="tip_amount",
        dataType=DecimalType(10,2),
        nullable=True,
        metadata={
            "comment": "Gratuity amount paid by the passenger"
        }
    ),
    StructField(
        name="tolls_amount",
        dataType=DecimalType(10,2),
        nullable=True,
        metadata={
            "comment": "Total toll charges incurred during the trip"
        }
    ),
    StructField(
        name="improvement_surcharge",
        dataType=DecimalType(10,2),
        nullable=True,
        metadata={
            "comment": "Regulatory surcharge supporting taxi industry improvement initiatives"
        }
    ),
    StructField(
        name="total_amount",
        dataType=DecimalType(10,2),
        nullable=True,
        metadata={
            "comment": "Total amount paid by the passenger including fare, taxes, tolls, surcharges, and tip"
        }
    ),
    StructField(
        name="total_amount_group",
        dataType=StringType(),
        nullable=True,
        metadata={
            "comment": "Total amount paid by the passenger grouped into 10er steps until 50. Can include nulls if amount should be negative."
        }
    ),
    StructField(
        name="congestion_surcharge",
        dataType=DecimalType(10,2),
        nullable=True,
        metadata={
            "comment": "Congestion surcharge applied to eligible trips operating in designated congestion zones"
        }
    ),
    StructField(
        name="airport_fee",
        dataType=DecimalType(10,2),
        nullable=True,
        metadata={
            "comment": "Airport access fee applied to eligible airport-related trips"
        }
    ),
    StructField(
        name="cbd_congestion_fee",
        dataType=DecimalType(10,2),
        nullable=True,
        metadata={
            "comment": "Central Business District congestion pricing fee applied to qualifying trips"
        }
    ),
])

## ETL

In [0]:
@dp.materialized_view(
    # Name der Zieltabelle
    name="analytics.gold.trips",
    # Beschreibung der Tabelle
    comment="This table shows the fakts in terms of the taken trips",
    # Liquid Clustering (Statt partitioning und Z-Order)
    cluster_by=[],
    cluster_by_auto=True,
    schema=schema,
)
def trips():
    df = spark.read.table("analytics.silver.trips_trn_trips")
    df_payment = spark.read.table("analytics.silver.payment_stm_payment")
    df_rate = spark.read.table("analytics.silver.payment_stm_rate")

    # Add Duration
    df = df.withColumn("duration", F.timestamp_diff("MINUTE",F.col("pickup_datetime"), F.col("dropoff_datetime")))
    # Remove trips with wrong Data
    df = df.filter(F.col("duration") >= 0)
    
    # Join in the description of Payment
    df = df.join(
        other=df_payment,
        on="payment_type",
        how="left",
    ).select(
        df["*"],
        df_payment["payment_description"]
    )

    # Join in the description of Rate
    df = df.join(
        other=df_rate,
        on="rate_code_id",
        how="left",
    ).select(
        df["*"],
        df_rate["rate_code_description"]
    )

    # Add Day of Trip
    df = df.withColumn("day_of_trip", F.to_date(F.col("pickup_datetime")))

    # Genarate Groups of total Amount
    df = df.withColumn(
        "total_amount_group",
        F.when(F.col("total_amount") < 10, "0-9")
        .when(F.col("total_amount") < 20, "10-19")
        .when(F.col("total_amount") < 30, "20-29")
        .when(F.col("total_amount") < 40, "30-39")
        .when(F.col("total_amount") < 50, "40-49")
        .when(F.col("total_amount") >= 50, "50+")
        .otherwise(None)
    )
    
    schema_columns = [(field.name, field.dataType) for field in schema.fields]
    df = df.select([F.col(col_name).cast(col_dtype) for col_name, col_dtype in schema_columns])
        
    return df